# End-to-End Trace: Research Agent + Cortex

Traces one real research episode through every stage (Cortex retrieve → Qwen config →
GPT Researcher → Grader/Reviser → Reflection → Cortex write-back), then runs a **second**
related episode to prove Cortex memory actually gets *used* on retrieval, not just written.

Requires a running Ollama server with a Qwen model pulled (confirmed working with `qwen3:14b`).
Each research episode is a real multi-minute run on a local model — this notebook makes real
calls, nothing here is mocked or dry-run.

In [1]:
import sys, os, asyncio, json, sqlite3
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "cortex").exists():
    REPO_ROOT = REPO_ROOT.parent  # in case the notebook is opened from a subdirectory
sys.path.insert(0, str(REPO_ROOT))

# Qwen via local Ollama -- adjust QWEN_MODEL if you've pulled a different one (`ollama list`)
os.environ["QWEN_API_KEY"] = "ollama"
os.environ["QWEN_BASE_URL"] = "http://localhost:11434/v1"
os.environ["QWEN_MODEL"] = "qwen3:14b"
os.environ["RETRIEVER"] = "duckduckgo"

# Fresh, isolated Cortex store + workspace for this trace, so it's reproducible
# and doesn't mix with any other run's memory.
TRACE_DIR = REPO_ROOT / "notebook_trace_data"
CORTEX_DIR = str(TRACE_DIR / "cortex_data")
WORKSPACE_DIR = str(TRACE_DIR / "workspace")
print("Repo root:", REPO_ROOT)
print("Cortex dir:", CORTEX_DIR)


Repo root: /Users/akhil_p/Desktop/personal/coding/self_healing_AI/self_healing_ai
Cortex dir: /Users/akhil_p/Desktop/personal/coding/self_healing_AI/self_healing_ai/notebook_trace_data/cortex_data


## Stage 1 — Cortex retrieve (cold start)

First episode, nothing in memory yet — this should come back empty.

In [2]:
from cortex import CortexStore, MemoryGraph, retrieve

QUESTION_1 = "What is the boiling point of water in Celsius at sea level?"

store = CortexStore(persist_dir=CORTEX_DIR)
graph = MemoryGraph(persist_path=str(Path(CORTEX_DIR) / "graph.sqlite3"))

retrieved_1 = retrieve(store, graph, QUESTION_1, scope="default")
print(retrieved_1.to_prompt_text())


(no prior memory — this is effectively a cold start)


## Stage 2 — Qwen configuration

Confirms which provider path got selected (`dashscope` vs `vllm_openai` vs `dry_run`) before any real call happens.

In [3]:
from agents.research import configure_qwen

provider = configure_qwen()
print("Provider selected:", provider)
assert provider != "dry_run", "QWEN_API_KEY/QWEN_BASE_URL not picked up -- check Ollama is running"
print("FAST_LLM   =", os.environ.get("FAST_LLM"))
print("SMART_LLM  =", os.environ.get("SMART_LLM"))
print("STRATEGIC_LLM =", os.environ.get("STRATEGIC_LLM"))


Provider selected: vllm_openai
FAST_LLM   = vllm_openai:qwen3:14b
SMART_LLM  = vllm_openai:qwen3:14b
STRATEGIC_LLM = vllm_openai:qwen3:14b


## Stage 3 — Fold memory into the query, run GPT Researcher

`context=` on the GPTResearcher constructor gets overwritten by `conduct_research()` (confirmed by reading `agent.py`), so memory gets folded into the query text itself -- the one injection point every internal stage actually reads.

In [4]:
from gpt_researcher import GPTResearcher

memory_context_1 = retrieved_1.to_prompt_text()
research_query_1 = QUESTION_1
if "no prior memory" not in memory_context_1:
    research_query_1 = (
        f"{QUESTION_1}\n\nRelevant context from prior research episodes -- do not "
        f"repeat work already covered, and treat Caution entries as known dead "
        f"ends:\n{memory_context_1}"
    )
print("Research query sent to GPT Researcher:\n")
print(research_query_1)


Research query sent to GPT Researcher:

What is the boiling point of water in Celsius at sea level?


In [5]:
researcher_1 = GPTResearcher(query=research_query_1, report_type="research_report")
await researcher_1.conduct_research()
print("conduct_research() done.")


Failed to import MCPRetriever: No module named 'langchain_mcp_adapters'


INFO:     [18:50:32] 🔍 Starting the research task for 'What is the boiling point of water in Celsius at sea level?'...


INFO:     [18:50:32] 🧪 Science Agent


INFO:     [18:50:32] 🌐 Browsing the web to learn more about the task: What is the boiling point of water in Celsius at sea level?...


INFO:     [18:50:33] 🤔 Planning the research strategy and subtasks...


...

INFO:     [18:50:54] 🌐 Scraping complete


INFO:     [18:50:54] 📚 Getting relevant content based on query: What is the Celsius boiling point of water at sea level?...


INFO:     [18:50:54] 📚 Combined research context: 0 MCP sources, web content


INFO:     [18:50:54] Finalized research step.
💸 Total Research Costs: $0.013927499999999999


conduct_research() done.


In [6]:
draft_1 = await researcher_1.write_report()
print(f"Draft length: {len(draft_1)} chars\n")
print(draft_1[:1500], "...\n[truncated for display]")


INFO:     [18:50:54] ✍️ Writing report for 'What is the boiling point of water in Celsius at sea level?'...


# The Boiling Point of Water at Sea Level: A Scientific Perspective  




## Introduction  




The boiling point of water is a fundamental concept in physics and chemistry, yet its precise value at sea level is often misunderstood. While many sources cite 100°C (212°F) as the boiling point of water at standard atmospheric pressure, recent scientific analyses and empirical measurements reveal a more nuanced picture. According to a comprehensive article from Complete Era Answers, the true boiling point of water at sea level is **99.9°C (211.8°F)**, not the commonly accepted 100°C ([url website](https://answers.completeera.com/post/what-s-the-true-boiling-point-at-sea-level-and-why-it-matters)). This slight deviation from the theoretical value stems from real-world variations in atmospheric pressure, humidity, and the purity of the water being boiled. Understanding this discrepancy is critical for applications ranging from cooking and scientific experimentation to meteorological studies.  




## The Science Behind Boiling Points  




...

The boiling point of water at sea level is not an absolute value but a dynamic equilibrium influenced by atmospheric pressure, humidity, and water purity. While the commonly cited value of 100°C is a useful approximation, the true boiling point is **99.9°C (211.8°F)**, as confirmed by empirical measurements and scientific analysis. This distinction has practical implications for cooking, scientific research, and environmental monitoring. Understanding these nuances not only enhances our grasp of fundamental physical principles but also ensures accuracy in real-world applications.  




### References  




- Complete Era Answers. (n.d.). *What’s the true boiling point at sea level and why it matters?* Retrieved from [https://answers.completeera.com/post/what-s-the-true-boiling-point-at-sea-level-and-why-it-matters](https://answers.completeera.com/post/what-s-the-true-boiling-point-at-sea-level-and-why-it-matters)


INFO:     [18:52:58] 📝 Report written for 'What is the boiling point of water in Celsius at sea level?'


Draft length: 8762 chars

# The Boiling Point of Water at Sea Level: A Scientific Perspective  

## Introduction  

The boiling point of water is a fundamental concept in physics and chemistry, yet its precise value at sea level is often misunderstood. While many sources cite 100°C (212°F) as the boiling point of water at standard atmospheric pressure, recent scientific analyses and empirical measurements reveal a more nuanced picture. According to a comprehensive article from Complete Era Answers, the true boiling point of water at sea level is **99.9°C (211.8°F)**, not the commonly accepted 100°C ([url website](https://answers.completeera.com/post/what-s-the-true-boiling-point-at-sea-level-and-why-it-matters)). This slight deviation from the theoretical value stems from real-world variations in atmospheric pressure, humidity, and the purity of the water being boiled. Understanding this discrepancy is critical for applications ranging from cooking and scientific experimentation to met

## Stage 4 — Grader

Independent check: does the draft actually support every claim, address the question, and cite real sources?

In [7]:
from agents.research.reflection import grade

verdict_1, g_in, g_out = await grade(QUESTION_1, draft_1)
print("Verdict:", verdict_1.verdict)
print("Mistakes found:", verdict_1.mistakes or "(none)")
print(f"Grader call cost: {g_in} in / {g_out} out tokens")


Verdict: fail
Mistakes found: ['(1): The draft claims the boiling point is **99.9°C** at sea level, citing "Complete Era Answers," but this is not a widely accepted scientific value. The standard boiling point of water at sea level is **100°C** under standard atmospheric pressure (101.325 kPa), as per all major scientific references (e.g., IUPAC, textbooks, and peer-reviewed journals). The specific claim of 99.9°C is not supported by credible sources and appears to be an outlier or misinterpretation.  ', '(3): All citations in the draft point to the same URL ([url website](https://answers.completeera.com/post/what-s-the-true-boiling-point-at-sea-level-and-why-it-matters)), which is not a peer-reviewed or authoritative scientific source. This URL is likely a placeholder or fabricated, making it impossible to verify the claims.  ', '(2): The draft focuses heavily on factors like humidity, water purity, and altitude but does not clearly answer the research question. While these factors ar

## Stage 4b — Reviser loop (only runs if the Grader asked for a revision)

Mirrors `research_pipeline.py`'s actual loop, including the hard policy cap that exists independently of `MAX_REVISIONS`.

In [8]:
from agents.research.reflection import revise
from agents.research.policies import DEFAULT_POLICIES, PolicyState, evaluate, Decision

MAX_REVISIONS = 3
policy_state = PolicyState()
healer_attempts = []
revision_n = 0
verdict = verdict_1
draft_1_final = draft_1
total_in, total_out = g_in, g_out

while verdict.verdict != "pass" and revision_n < MAX_REVISIONS:
    policy_result = evaluate(DEFAULT_POLICIES, policy_state)
    if policy_result.decision in (Decision.DENY, Decision.ASK):
        print("Policy stopped the loop:", policy_result.reason)
        break
    print(f"--- Revision {revision_n + 1} ---")
    draft_1_final, r_in, r_out = await revise(draft_1_final, verdict.mistakes)
    total_in += r_in; total_out += r_out
    healer_attempts.append({"mistakes": verdict.mistakes, "revision": revision_n + 1})
    verdict, g_in, g_out = await grade(QUESTION_1, draft_1_final)
    total_in += g_in; total_out += g_out
    print("New verdict:", verdict.verdict, "| mistakes:", verdict.mistakes or "(none)")
    revision_n += 1
    policy_state.revision_count = revision_n
    policy_state.total_input_tokens, policy_state.total_output_tokens = total_in, total_out

print(f"\nFinal verdict: {verdict.verdict} after {revision_n} revision(s)")


--- Revision 1 ---


New verdict: revise | mistakes: ['(3): Citation "Journal of Chemical Education" and "General principles of thermodynamics and physical chemistry" lack specific URLs or page numbers, making it impossible to verify their support for claims about boiling point variations.  ', '(1): The claim about distilled water boiling at "0.5°C higher" than tap water is not cited with a specific source or URL, though it is a common scientific assertion.  ', '(3): The reference to "standard scientific references" is too vague to plausibly support claims about boiling point fundamentals.']
--- Revision 2 ---


New verdict: revise | mistakes: ['(1): Claim about variations in boiling point due to impurities cited with invalid DOI (https://doi.org/10.1000/xyz456) – this DOI does not exist and cannot plausibly support the claim.  ', '(1): Multiple sections cite the same placeholder DOI (https://doi.org/10.1000/xyz123) for different claims, which is not credible.  ', '(3): URLs like https://example.com/thermodynamics-reference in the references are clearly fake and do not plausibly support any claims.  ', '(2): While the question is addressed directly (boiling point is 100°C), the report spends excessive detail on tangential topics (e.g., cooking, altitude tables) without explicitly tying them back to the core question.']
--- Revision 3 ---


New verdict: pass | mistakes: ['None detected. The draft directly answers the research question with the correct value (100°C) and cites reputable sources (even though URLs are not provided, the sources themselves are plausible). All claims are consistent with scientific consensus, and no contradictions or unaddressed aspects of the question are present.']

Final verdict: pass after 3 revision(s)


## Stage 5 — Reflection

Produces the three fields Cortex actually persists for future episodes to learn from.

In [9]:
from agents.research.reflection import reflect
from cortex import Outcome

outcome_1 = Outcome.SUCCESS if verdict.verdict == "pass" else (
    Outcome.PARTIAL if revision_n > 0 else Outcome.FAILURE
)
summary_text_1 = f"research: {draft_1_final[:150]}\ngrade: final_verdict={verdict.verdict} revisions={revision_n}"
reflection_1, r_in, r_out = await reflect(QUESTION_1, outcome_1.value, summary_text_1)
total_in += r_in; total_out += r_out

print("Outcome:", outcome_1.value)
print("abstracted_pattern:", reflection_1.abstracted_pattern)
print("failure_summary:", reflection_1.failure_summary)
print("distilled_learnings:", reflection_1.distilled_learnings)


Outcome: success
abstracted_pattern: Leveraging well-established scientific principles and standard reference conditions (e.g., standard atmospheric pressure at sea level) to confirm widely known physical properties.
failure_summary: None
distilled_learnings: ['Standard conditions (e.g., 1 atm pressure) are critical for defining physical property measurements.  ', 'Reliable sources often directly reference universally accepted scientific constants.  ', 'Simple, targeted research can efficiently resolve common scientific questions.']


## Stage 6 — Cortex write-back

Persists the episode. `recall_count`/`intuition_count` should go from 0 to 1.

In [10]:
from cortex import Episode, write_back
from cortex.schema import ActionArcStep, CostActual, HealerAttempt

print("Cortex stats BEFORE write-back:", store.stats())

episode_1 = Episode(
    domain="research",
    scope="default",
    problem_summary=QUESTION_1,
    action_arc=[ActionArcStep(action="research", tool_used="GPTResearcher", output=draft_1_final[:300])],
    healer_attempts=[HealerAttempt(hypothesis="; ".join(h["mistakes"]) or "unspecified",
                                    fix=f"revision {h['revision']}", outcome="revise") for h in healer_attempts],
    outcome=outcome_1,
    final_solution=draft_1_final if outcome_1 != Outcome.FAILURE else None,
    abstracted_pattern=reflection_1.abstracted_pattern,
    failure_summary=reflection_1.failure_summary,
    distilled_learnings=reflection_1.distilled_learnings,
    cost_actual=CostActual(input_tokens=total_in, output_tokens=total_out),
    healer_attempt_count=revision_n,
)

write_back(store, graph, episode_1, retrieved=retrieved_1)
print("\nCortex stats AFTER write-back:", store.stats())


Cortex stats BEFORE write-back: {'recall_count': 0, 'intuition_count': 0, 'caution_count': 0, 'instinct_count': 0, 'total_episodes': 0}



Cortex stats AFTER write-back: {'recall_count': 1, 'intuition_count': 1, 'caution_count': 0, 'instinct_count': 0, 'total_episodes': 1}


## Stage 7 — Inspect the SQLite-backed memory graph directly

Proves persistence is real, queryable disk state, not an in-memory illusion.

In [11]:
graph_db_path = Path(CORTEX_DIR) / "graph.sqlite3"
print("Graph DB file exists:", graph_db_path.exists(), "--", graph_db_path)

conn = sqlite3.connect(graph_db_path)
print("\nNodes:")
for row in conn.execute("SELECT node_id, node_type FROM nodes"):
    print(" ", row)
print("\nEdges:")
for row in conn.execute("SELECT source, target, edge_type, weight FROM edges"):
    print(" ", row)
conn.close()


Graph DB file exists: True -- /Users/akhil_p/Desktop/personal/coding/self_healing_AI/self_healing_ai/notebook_trace_data/cortex_data/graph.sqlite3

Nodes:
  ('4dae3462-2458-488e-8ecb-3838b58aac7f', 'episode')
  ('pattern::7260796158140877512', 'pattern')

Edges:
  ('4dae3462-2458-488e-8ecb-3838b58aac7f', 'pattern::7260796158140877512', 'instance_of', 0.5)


## Stage 8 — Second episode: does memory actually get *used*?

A related question, close enough that Recall/Intuition should surface episode 1. This is the real test of "memory," not just "logging."

In [12]:
QUESTION_2 = "At what temperature does water boil, measured in Celsius, under normal atmospheric pressure?"

retrieved_2 = retrieve(store, graph, QUESTION_2, scope="default")
print(retrieved_2.to_prompt_text())
print("\nRecall hits:", len(retrieved_2.recall_hits))
print("Intuition hits:", len(retrieved_2.intuition_hits))


## Recall — similar past tasks (literal match)
- [success] What is the boiling point of water in Celsius at sea level?

## Intuition — this resembles a deeper pattern
- Leveraging well-established scientific principles and standard reference conditions (e.g., standard atmospheric pressure at sea level) to confirm widely known physical properties.

## Intuition (associative) — 1 related episode(s) surfaced via memory graph, not direct similarity

Recall hits: 1
Intuition hits: 1


If Recall or Intuition surfaced episode 1 above, memory is doing real work across episodes, not just accumulating a write-only log. Run the full pipeline for question 2 the same way as Stage 3 onward if you want the complete second trace, or use `research_pipeline.run_research_pipeline(QUESTION_2, cortex_dir=CORTEX_DIR, workspace_dir=WORKSPACE_DIR)` for the one-shot version of everything above.

## What this notebook does *not* cover

- **RSI (Spec 5)**: `rsi/` and `selfmod/` are separate, slow, multi-minute-per-cycle
  processes (a harnessing-level cycle benchmarks 5 tasks twice — baseline and candidate).
  Run them directly: `python cli.py --self-improve --harness research`.
- **The coding harness**: `pipeline.py` / `agents/base.py` / `agents/pipeline_agents.py`,
  Anthropic-backed, untouched by anything in this notebook.
- **Policy DENY in practice**: the revision loop above includes the policy check, but
  the default caps (6 revisions, 900s, 500k tokens) are generous enough that a normal
  run won't trigger one — see `agents/research/policies.py` to lower them if you want
  to see a DENY fire.